In [1]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

In [5]:

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [7]:
import langchain.mcp

C:\Users\socgen\AppData\Local\Temp\ipykernel_7404\1520507502.py:1: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  import langchain.mcp


In [8]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()

Overwriting cinebot_mcp_server.py


In [9]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

In [10]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("C:\\Users\\socgen\\ML\\mcp_4_agents\\MCP in Langchain\\cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


### Call a tool without AI or agent

In [14]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    showtimes_tool = next((t for t in tools if t.name == "check_showtimes"), None)
    direct_result = await showtimes_tool.ainvoke({"movie_title": "Interstellar"})
    print("Direct result from MCP tool:", direct_result)

    try:
        cinebot_agent = create_agent(
            model=model_basic, tools=tools)
        result = await cinebot_agent.ainvoke(
            {"messages": [{"role": "user", "content": "What are the showtimes for Interstellar?"}]}
        )
        print("Agent result:", result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct result from MCP tool: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_5fe48c37-4d55-479a-b382-5367bb40bdcc'}]
Agent result: {'messages': [HumanMessage(content='What are the showtimes for Interstellar?', additional_kwargs={}, response_metadata={}, id='d2b83fec-9a43-41fc-845e-5481e4ab4519'), AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'poolside/laguna-s-2.1:free', 'id': 'gen-1790997486-ksaOSjFLSU874T2DZevG', 'created': 1790997486, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a0ffc4-ccc5-7b52-8180-8f118906c652-0', tool_calls=[{'name': 'check_showtimes', 'args': {'movie_title': 'Interstellar'}, 'id': 'chatcmpl-tool-cb8779a3563f49a3b9258ca8c125f133', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input